# Baseline RAG query pipeline

Embed the user query → **top‑k** semantic search on the S3 Vector index (`rag-vector-2`) → resolve chunk text directly from **vector metadata** stored in `rag-vector-2` via `get_vectors()`.

| Step | What happens |
|---|---|
| 1 | Same encoder as ingest (`EMBED_MODEL`, default `intfloat/multilingual-e5-small`) |
| 2 | Query prefixed with `query: ` (E5; corpus used `passage: ` in `full_embeddings.py`) |
| 3 | `query_vectors` on `rag-vector-2` → returns top-k keys + distances |
| 4 | `get_vectors()` on unresolved keys → pulls chunk text from vector metadata |

**No local `kb/` files required.** Chunk text is stored alongside the vectors in `rag-vector-2`.

**Credentials:** `VECTORS_AWS_ACCESS_KEY_ID`, `VECTORS_AWS_SECRET_ACCESS_KEY`, and region (see `src/utils/aws_profiles.py`).


## 1. Dependencies (run once per environment)

In [9]:
%pip install sentence-transformers boto3 pandas --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Project path and imports

In [10]:
import json
import os
import sys
from pathlib import Path

import pandas as pd


def project_root() -> Path:
    """Directory that contains `src/` (works if cwd is repo root or `rag/`)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "src" / "retrieval").is_dir():
            return p
    raise RuntimeError("Could not find project root (folder with src/retrieval). cd to the repo root or rag/.")


_ROOT = project_root()
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

from src.retrieval.s3_vectors_rag import S3VectorsRAGRetriever

print(f"Project root: {_ROOT}")

Project root: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469


## 3. Configuration

Override with environment variables: `RAG_VECTOR_BUCKET`, `RAG_VECTOR_INDEX`, `RAG_KB_DIR`, `VECTORS_AWS_DEFAULT_REGION`, `EMBED_MODEL`.

In [11]:
TOP_K = 5

VECTOR_BUCKET = os.environ.get("RAG_VECTOR_BUCKET", "is469-genai-grp-project")
INDEX_NAME = os.environ.get("RAG_VECTOR_INDEX", "rag-vector-2")
REGION = os.environ.get("VECTORS_AWS_DEFAULT_REGION", "ap-southeast-1")

_kb_env = os.environ.get("RAG_KB_DIR", "").strip()
KB_DIR = Path(_kb_env) if _kb_env else (_ROOT / "kb")

print(f"Bucket: {VECTOR_BUCKET}")
print(f"Index:  {INDEX_NAME}")
print(f"Region: {REGION}")
print(f"kb_dir: {KB_DIR} (exists={KB_DIR.is_dir()})")

Bucket: is469-genai-grp-project
Index:  rag-vector-2
Region: ap-southeast-1
kb_dir: c:\Users\Jonathan\Desktop\Y4S2\GenAI\is469\kb (exists=True)


## 3b. Build S3 Vectors client for chunk text resolution

Uses `s3vectors_client` (same credentials as the retriever) to call `get_vectors()` on the keys
returned by the query. Chunk text is resolved directly from the vector metadata stored in `rag-vector-2` —
no local `kb/` file or separate S3 bucket required.


In [12]:
from src.utils.aws_profiles import s3vectors_client

# Same bucket + index as the retriever
vectors_client = s3vectors_client(region_name=REGION)

def resolve_chunks_from_s3(keys: list[str]) -> dict[str, str]:
    """
    Fetch full vector metadata for a list of keys from rag-vector-2.
    Returns a dict of {key: chunk_text}.
    """
    if not keys:
        return {}
    response = vectors_client.get_vectors(
        vectorBucketName=VECTOR_BUCKET,
        indexName=INDEX_NAME,
        keys=keys,
        returnData=False,       # We only need metadata, not the embedding floats
        returnMetadata=True,
    )
    result = {}
    for vec in response.get("vectors", []):
        key = vec.get("key")
        meta = vec.get("metadata") or {}
        # Try common field names the ingest script may have used
        text = (
            meta.get("text")
            or meta.get("chunk_text")
            or meta.get("content")
        )
        if key and text:
            result[key] = text
    return result

print("✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata")


✓ s3vectors_client ready — chunk text will be resolved from rag-vector-2 metadata


## 4. Build retriever (loads embedder on first `retrieve`)

The next cell replaces `s3_vectors_rag._guess_kb_paths` so metadata names match **this repo’s** `kb/` files (e.g. `eng_jap_chunks_embedded_full.jsonl` in metadata → `eng_jap_chunks_embedded.jsonl` on disk, and no double `.jsonl.jsonl` when metadata already includes `.jsonl`).

In [13]:
import src.retrieval.s3_vectors_rag as _s3_rag


def _guess_kb_paths(kb_dir: Path, source_file: str) -> list[Path]:
    """Map vector metadata `source_file` to JSONL paths under kb/ (repo layout)."""
    raw = (source_file or "").strip()
    if not raw:
        return []
    name = Path(raw).name
    if name.endswith(".jsonl"):
        basename = name
        stem_no_ext = Path(name).stem
    else:
        stem_no_ext = name
        basename = f"{name}.jsonl"
    candidates: list[Path] = []
    seen: set[Path] = set()

    def add(p: Path) -> None:
        if p not in seen:
            seen.add(p)
            candidates.append(p)

    add(kb_dir / basename)
    add(kb_dir / f"{stem_no_ext}_vectors.jsonl")
    if stem_no_ext.endswith("_embedded_full"):
        add(kb_dir / f"{stem_no_ext[:-len('_full')]}.jsonl")
    if stem_no_ext.endswith("_embedded") and not stem_no_ext.endswith("_embedded_full"):
        add(kb_dir / f"{stem_no_ext}_full.jsonl")

    return [p for p in candidates if p.is_file()]


_s3_rag._guess_kb_paths = _guess_kb_paths

retriever = S3VectorsRAGRetriever(
    vector_bucket_name=VECTOR_BUCKET,
    index_name=INDEX_NAME,
    region_name=REGION,
    kb_dir=KB_DIR,
    top_k=TOP_K,
)

print("Retriever ready (encoder loads lazily on first retrieve).")

Retriever ready (encoder loads lazily on first retrieve).


## 5. Run retrieval

Set `USER_QUERY` to the student’s text, then run the next cell.

In [14]:
USER_QUERY = "Example: How do I express polite requests in Japanese?"

context, chunks = retriever.retrieve(USER_QUERY)

# ── Resolve unresolved chunks from rag-vector-2 metadata ────────────────────
unresolved_keys = [
    c.key for c in chunks
    if (c.text or "").startswith("(no local text resolved")
]

if unresolved_keys:
    print(f"Resolving {len(unresolved_keys)} chunk(s) from rag-vector-2...")
    s3_text_map = resolve_chunks_from_s3(unresolved_keys)
    for chunk in chunks:
        if (chunk.text or "").startswith("(no local text resolved"):
            chunk.text = s3_text_map.get(chunk.key, chunk.text)

# ── Format context string ────────────────────────────────────────────────────
context = "\n---\n".join(
    f"[{c.source_file} L{c.source_line}]\n{c.text or '(unresolved)'}"
    for c in chunks
)

print("=== Formatted context (for prompting) ===\n")
print(context)

# ── DataFrame summary ────────────────────────────────────────────────────────
_preview_chars = 320
_rows: list[dict] = []
for i, c in enumerate(chunks, start=1):
    unresolved = (c.text or "").startswith("(no local text resolved")
    t = c.text or ""
    preview = t if len(t) <= _preview_chars else f"{t[: _preview_chars - 1]}…"
    _rows.append(
        {
            "rank": i,
            "distance": c.distance,
            "key": c.key,
            "source_file": c.source_file,
            "source_line": c.source_line,
            "resolved": not unresolved,
            "text_preview": preview,
            "chunk_text": t,
        }
    )

hits_df = pd.DataFrame(_rows)
if hits_df["distance"].notna().any():
    hits_df["distance"] = pd.to_numeric(hits_df["distance"], errors="coerce").round(6)

hits_summary = hits_df[
    ["rank", "distance", "key", "source_file", "source_line", "resolved", "text_preview"]
]
print("\n=== Retrieval hits (summary DataFrame: `hits_df`, display below) ===\n")
hits_summary


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: intfloat/multilingual-e5-small
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Resolving 4 chunk(s) from rag-vector-2...
=== Formatted context (for prompting) ===

[eng_jap_chunks_embedded_full.jsonl L8432]
(no local text resolved for key=eng_jap_chunks_embedded_full:8431; source_file=eng_jap_chunks_embedded_full.jsonl line=8432)
---
[eng_jap_chunks_embedded_full.jsonl L79]
[1] EN: I'm tired.
[1] JA: 私は、疲れています。

[2] EN: Who wants some hot chocolate?
[2] JA: ホットチョコレート欲しい人ー？

[3] EN: Speak more slowly, please!
[3] JA: もっとゆっくり話してください！

[4] EN: When do we arrive?
[4] JA: いつ着くの？

[5] EN: The check, please.
[5] JA: 勘定お願いします。

[6] EN: The check, please.
[6] JA: 勘定書をお願いします。

[7] EN: The check, please.
[7] JA: 勘定を頼むよ。

[8] EN: The check, please.
[8] JA: お勘定して下さい。
---
[eng_jap_chunks_embedded_full.jsonl L30252]
(no local text resolved for key=eng_jap_chunks_embedded_full:30251; source_file=eng_jap_chunks_embedded_full.jsonl line=30252)
---
[eng_jap_chunks_embedded_full.jsonl L39469]
(no local text resolved for key=eng_jap_chunks_embedded_full:39468; source_file=eng_jap_chu

,rank,distance,key,source_file,source_line,resolved,text_preview
0,1,0.103171,eng_jap_chunks_embedded_full:8431,eng_jap_chunks_embedded_full.jsonl,8432,False,(no local text resolved for key=eng_jap_chunks...
1,2,0.104292,eng_jap_chunks_embedded_full:78,eng_jap_chunks_embedded_full.jsonl,79,True,[1] EN: I'm tired.\n[1] JA: 私は、疲れています。\n\n[2] ...
2,3,0.105754,eng_jap_chunks_embedded_full:30251,eng_jap_chunks_embedded_full.jsonl,30252,False,(no local text resolved for key=eng_jap_chunks...
3,4,0.106427,eng_jap_chunks_embedded_full:39468,eng_jap_chunks_embedded_full.jsonl,39469,False,(no local text resolved for key=eng_jap_chunks...
4,5,0.107059,eng_jap_chunks_embedded_full:38881,eng_jap_chunks_embedded_full.jsonl,38882,False,(no local text resolved for key=eng_jap_chunks...


In [16]:
# Verify eng_jap_chunks.jsonl can resolve our known unresolved lines
import json
from pathlib import Path

chunks_file = KB_DIR / "eng_jap_chunks.jsonl"
print(f"File exists: {chunks_file.exists()}")

with open(chunks_file, encoding="utf-8") as f:
    lines = f.readlines()

print(f"Total lines: {len(lines):,}")

# Check the exact line numbers we know are unresolved
for lineno in [8432, 30252, 39469, 38882]:
    if lineno - 1 < len(lines):
        record = json.loads(lines[lineno - 1])
        print(f"\nLine {lineno}: {json.dumps(record, ensure_ascii=False)[:200]}")
    else:
        print(f"\nLine {lineno}: OUT OF RANGE")

File exists: True
Total lines: 46,616

Line 8432: {"chunk_id": "engjap_chunk_050586_050593", "start_row": 50586, "end_row": 50593, "n_pairs": 8, "chunk_text": "[1] EN: I'm very pleased to meet you.\n[1] JA: お目にかかれて、たいへんうれしいです。\n\n[2] EN: Could you gi

Line 30252: {"chunk_id": "engjap_chunk_181506_181513", "start_row": 181506, "end_row": 181513, "n_pairs": 8, "chunk_text": "[1] EN: Express yourself as you please!\n[1] JA: 自由に意見を述べてください。\n\n[2] EN: Take a shower

Line 39469: {"chunk_id": "engjap_chunk_236808_236815", "start_row": 236808, "end_row": 236815, "n_pairs": 8, "chunk_text": "[1] EN: I now want to drink beer.\n[1] JA: 今、ビールが飲みたい。\n\n[2] EN: Another pillow, please

Line 38882: {"chunk_id": "engjap_chunk_233286_233293", "start_row": 233286, "end_row": 233293, "n_pairs": 8, "chunk_text": "[1] EN: Let's meet at the station.\n[1] JA: 駅で待ち合わせしようね。\n\n[2] EN: I don't know. Please
